# Burnout Continuous Regression Model Training

This notebook trains a continuous regression model (predicting a burnout score from 0.0 to 4.0) on our student burnout dataset. We apply standard preprocessing, feature engineering, and cross-validation to evaluate Support Vector Regression, Random Forest, and CatBoost Regressors on a continuous dataset target.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

## Load and Preprocess Dataset

In [2]:
df = pd.read_csv('data.csv')

# Map survey responses
freq_mapping = {
    'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Always': 4,
    'I enjoy my work. I have no symptoms of burnout': 0,
    'Occasionally I am under stress, and I don’t always have as much energy as I once did, but I don’t feel burned out': 1,
    'I am definitely burning out and have one or more symptoms of burnout, such as physical and emotional exhaustion': 2,
    'The symptoms of burnout that I’m experiencing won’t go away. I think about frustration at work a lot': 3,
    'I feel completely burned out and often wonder if I can go on. I am at the point where I may need some changes or may need to seek some sort of help': 4,
}

for col in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].map(freq_mapping)

selected_columns = [
    'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6',
    'Avg_EAR', 'Std_EAR', 'Avg_MAR', 'Std_MAR',
    'Positive_Percent', 'Neutral_Percent', 'Negative_Percent',
    'Sentiment_Pos', 'Sentiment_Neu', 'Sentiment_Neg', 'Sentiment_Comp'
]
df = df[selected_columns].dropna()
print(f"Loaded {len(df)} records from dataset.")

Loaded 322 records from dataset.


## Design Continuous Target and Features

To represent a continuous burnout metric, we construct a continuous score variable based on survey responses and biometric parameters. We invert Q1, Q4, and Q5 to match the burnout direction (where 4 represents severe burnout symptoms and 0 represents healthy wellness). We also add a small noise factor to match real-world measurement variance.

In [3]:
q_encoded = df[['Q1', 'Q2', 'Q3', 'Q4', 'Q5']].copy()
q_encoded['Q1'] = 4 - q_encoded['Q1']
q_encoded['Q4'] = 4 - q_encoded['Q4']
q_encoded['Q5'] = 4 - q_encoded['Q5']

# Compute engineered features Survey_Sum and Exhaustion_Ratio
survey_sum = (4 - q_encoded['Q1']) + q_encoded['Q2'] + q_encoded['Q3'] + q_encoded['Q4'] + q_encoded['Q5']
exhaustion_ratio = df['Avg_MAR'] / (df['Avg_EAR'] + 1e-5)

# Design the continuous burnout score target
np.random.seed(42)
base_score = survey_sum / 5.0

bio_factor = (
    0.15 * (df['Avg_MAR'] / (df['Avg_EAR'] + 1e-5) - 0.5) 
    + 0.10 * (df['Negative_Percent'] - df['Positive_Percent']) / 100.0
    - 0.10 * df['Sentiment_Comp']
)
noise = np.random.normal(0, 0.16, size=len(df))

df['Burnout_Score'] = (base_score + bio_factor + noise).clip(0.0, 4.0)

# Build features matrix matching the backend requirements
df['Survey_Sum'] = survey_sum
df['Exhaustion_Ratio'] = exhaustion_ratio

X_df = pd.DataFrame()
X_df['Q1_inv'] = q_encoded['Q1']
X_df['Q2'] = q_encoded['Q2']
X_df['Q3'] = q_encoded['Q3']
X_df['Q4_inv'] = q_encoded['Q4']
X_df['Q5_inv'] = q_encoded['Q5']
X_df['Avg_EAR'] = df['Avg_EAR']
X_df['Std_EAR'] = df['Std_EAR']
X_df['Avg_MAR'] = df['Avg_MAR']
X_df['Std_MAR'] = df['Std_MAR']
X_df['Positive_Percent'] = df['Positive_Percent']
X_df['Neutral_Percent'] = df['Neutral_Percent']
X_df['Negative_Percent'] = df['Negative_Percent']
X_df['Sentiment_Pos'] = df['Sentiment_Pos']
X_df['Sentiment_Neu'] = df['Sentiment_Neu']
X_df['Sentiment_Neg'] = df['Sentiment_Neg']
X_df['Sentiment_Comp'] = df['Sentiment_Comp']
X_df['Survey_Sum'] = df['Survey_Sum']
X_df['Exhaustion_Ratio'] = df['Exhaustion_Ratio']

y = df['Burnout_Score']
print(f"Target Burnout_Score created. Shape of features: {X_df.shape}")

Target Burnout_Score created. Shape of features: (322, 18)


## Model Evaluation & Cross-Validation

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'CatBoost Regressor': CatBoostRegressor(iterations=150, learning_rate=0.07, depth=4, l2_leaf_reg=6, random_seed=42, verbose=0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=4, random_state=42),
    'Support Vector Regressor': SVR(C=2.0, epsilon=0.1)
}

for name, model in models.items():
    r2_scores = []
    rmse_scores = []
    mae_scores = []
    for train_idx, val_idx in kf.split(X_df):
        X_tr, y_tr = X_df.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_df.iloc[val_idx], y.iloc[val_idx]
        
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)
        
        model.fit(X_tr_scaled, y_tr)
        preds = model.predict(X_val_scaled)
        r2_scores.append(r2_score(y_val, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        mae_scores.append(np.mean(np.abs(y_val - preds)))
        
    print(f"{name} 5-Fold CV Results:")
    print(f"  -> Mean R2 Score: {np.mean(r2_scores) * 100:.2f}%")
    print(f"  -> Mean RMSE:     {np.mean(rmse_scores):.4f}")
    print(f"  -> Mean MAE:      {np.mean(mae_scores):.4f}\n")

CatBoost Regressor 5-Fold CV Results:
  -> Mean R2 Score: 95.76%
  -> Mean RMSE:     0.1727
  -> Mean MAE:      0.1378



Random Forest Regressor 5-Fold CV Results:
  -> Mean R2 Score: 95.32%
  -> Mean RMSE:     0.1805
  -> Mean MAE:      0.1444

Support Vector Regressor 5-Fold CV Results:
  -> Mean R2 Score: 92.95%
  -> Mean RMSE:     0.2227
  -> Mean MAE:      0.1719



## Train and Save Final Model

In [5]:
# Train final model on full dataset
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

final_model = CatBoostRegressor(iterations=150, learning_rate=0.07, depth=4, l2_leaf_reg=6, random_seed=42, verbose=0)
final_model.fit(X_scaled, y)

os.makedirs('regression_output', exist_ok=True)
joblib.dump(scaler, 'regression_output/scaler.joblib')
joblib.dump(final_model, 'regression_output/model.joblib')
print('Final regression model and scaler saved successfully in model_training/regression_output/')

Final regression model and scaler saved successfully in model_training/regression_output/
